# Free Throw Model — Cross‑Session Evaluation & Calibration

Capabilities of this notebook:
- Run **cross-session** tests (train on one session, test on another).
- Try multiple **rebalancing** strategies (class weights, under/over-sampling, SMOTE).
- Evaluate multiple **models** with **GroupKFold** by session.
- **Calibrate** probabilities and **tune thresholds** for your cost function.
- Produce **confusion matrices**, **PR curves**, **reliability (calibration) curves**, and **CSV reports**.

> **Instructions**
> 1. Edit the paths in the **Config** cell to point to `X.csv` and `y.csv` for each session.
> 2. Optionally adjust the model list and rebalancing options.
> 3. Run cells top-to-bottom. Outputs will be saved next to each session's dataset under `analysis/datasets/experiments/`.
>
> **Assumptions**
> - `X.csv` is all-numeric feature columns (same schema across sessions).
> - `y.csv` contains a single column `label` with values like `made`/`miss` or `1`/`0` (we map to 1 for made, 0 for miss).
> - Optional column `clip_id` in `X.csv` will be used for error analysis if present.


NOTES FOR MYSELF:
* Rebalancing → fixes imbalance during training.
* Calibration (sigmoid/isotonic) → ensures probability outputs are trustworthy.
* Thresholding → picks decision cutoffs.

In [89]:
# ===== 0) SETUP / IMPORTS / CONFIG =====
from __future__ import annotations
import sys, re, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- robust repo root + import path setup ---
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for _ in range(10):
        if (p / ".git").exists() or (p / "scripts" / "utils" / "plots.py").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return (start or Path.cwd()).resolve()

REPO_ROOT = find_repo_root()
for path in (REPO_ROOT, REPO_ROOT / "scripts"):
    if str(path) not in sys.path:
        sys.path.append(str(path))

# --- sklearn ---
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    precision_recall_fscore_support, average_precision_score,
    roc_auc_score, brier_score_loss
)

# --- our plotting util (saves CM, PR, calibration) ---
from utils.plots import save_all_eval_figures

import importlib, utils.plots as up, inspect

# Force reload
importlib.reload(up)

# Re-import the function (important after reload)
from utils.plots import save_all_eval_figures

# Verify you're using the right file + new code
print("Using plots from:", up.__file__)
print(inspect.getsource(up.save_pr_curves)[:400])


# --- run directory helper ---
def start_run(label: str = "baseline") -> Path:
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    root = REPO_ROOT / "experiments"
    root.mkdir(parents=True, exist_ok=True)
    # auto-increment run index if we collide on timestamp+label
    existing = sorted(root.glob(f"{ts}_run-*_{label}"))
    idx = 1
    if existing:
        m = re.search(r"_run-(\d{3})_", existing[-1].name)
        idx = (int(m.group(1)) + 1) if m else 1
    rd = root / f"{ts}_run-{idx:03d}_{label}"
    (rd / "figures").mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Run directory: {rd}")
    return rd

run_dir = start_run("baseline")
CLASS_NAMES = ["neg", "pos"]  # binary

# --- project config (paths & models) ---
SESSIONS = {
    "session_02": (
        REPO_ROOT / "data/alton_overson/session_02/analysis/datasets/features/v1.0/X.csv",
        REPO_ROOT / "data/alton_overson/session_02/analysis/datasets/features/v1.0/y.csv",
        REPO_ROOT / "data/alton_overson/session_02/analysis/datasets",
    ),
    "session_03": (
        REPO_ROOT / "data/alton_overson/session_03/analysis/datasets/features/v1.0/X.csv",
        REPO_ROOT / "data/alton_overson/session_03/analysis/datasets/features/v1.0/y.csv",
        REPO_ROOT / "data/alton_overson/session_03/analysis/datasets",
    ),
}

POSITIVE_CLASS = 1
CALIBRATE = True
CALIB_METHOD = "sigmoid"       # safer on small data
REBALANCE  = "class_weight"    # or "none"

MODELS = {
    "LogisticRegression":  LogisticRegression(max_iter=2000),
    "LinearSVC":           LinearSVC(C=1.0, max_iter=10000, tol=1e-3),
    "RandomForest":        RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting":    GradientBoostingClassifier(random_state=42),
    "GaussianNB":          GaussianNB(),
    # "LinearSVC_prob":    SVC(kernel="linear", probability=True, max_iter=10000)
}


Using plots from: /Users/kwill55/RVL/personalized-sports-optimization/utils/plots.py
def save_pr_curves(y_true, y_proba, class_names, outdir: Path):
    outdir = _ensure_dir(outdir)
    n_classes = len(class_names)
    P = _as_prob_matrix(y_proba, n_classes)

    fig, ax = plt.subplots(figsize=(7,5), dpi=150)
    if n_classes == 2:
        # Binary: single curve for positive class
        prob_pos = P[:, 1]
        precision, recall, _ = precision_recall_curve(y_true, prob_pos)
  
[INFO] Run directory: /Users/kwill55/RVL/personalized-sports-optimization/experiments/2025-08-19_20-48-21_run-001_baseline


In [90]:
# ===== 1) TRAINING (with optional rebalancing via class_weight) =====

# Choose direction
train_session_name, test_session_name = "session_02", "session_03"

# Load & unpack
X_train_df, y_train, out_dir_train = None, None, None
X_test_df,  y_test,  out_dir_test  = None, None, None
for session_name, (X_path, y_path, out_root) in SESSIONS.items():
    X_df, y = load_features_and_labels(X_path, y_path)  # <- your helper
    if session_name == train_session_name:
        X_train_df, y_train, out_dir_train = X_df, y, out_root
    if session_name == test_session_name:
        X_test_df,  y_test,  out_dir_test  = X_df, y, out_root

# Align features & split arrays/meta
X_train_df, X_test_df, shared_cols, meta_cols = align_feature_columns(X_train_df, X_test_df)
X_train, X_train_meta = split_features_and_metadata(X_train_df, shared_cols, meta_cols)
X_test,  X_test_meta  = split_features_and_metadata(X_test_df,  shared_cols, meta_cols)

print("Train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test  class counts:", dict(zip(*np.unique(y_test,  return_counts=True))))

# Pick one model for this cell (you can loop elsewhere)
selected_model_name = "LogisticRegression"
base_estimator = MODELS[selected_model_name]

# Optional rebalancing
if (REBALANCE == "class_weight") and hasattr(base_estimator, "set_params"):
    try:
        base_estimator = base_estimator.set_params(class_weight="balanced")
    except Exception:
        pass

# Build & fit pipeline (scaler → clf)
training_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", base_estimator)])
trained_pipeline = training_pipeline.fit(X_train, y_train)


Train class counts: {0: 34, 1: 65}
Test  class counts: {0: 67, 1: 115}


In [92]:
# ===== 2) CALIBRATION (post-training probability calibration) =====

if CALIBRATE:
    # folds choice robust to small/imbalanced data
    minority_count = min(np.bincount(y_train)) if len(np.unique(y_train)) == 2 else 0
    n_splits = 3 if minority_count >= 3 else 2
    calibration_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    calibrated_model = CalibratedClassifierCV(
        estimator=trained_pipeline,
        method=CALIB_METHOD,
        cv=calibration_cv
    ).fit(X_train, y_train)
else:
    calibrated_model = trained_pipeline

# calibrated probabilities on TEST
if hasattr(calibrated_model, "predict_proba"):
    y_proba = calibrated_model.predict_proba(X_test)[:, 1]
else:
    # decision_function fallback (maps to (0,1) for ranking; not true calibration)
    try:
        decision_scores = calibrated_model.decision_function(X_test)
        y_proba = 1 / (1 + np.exp(-decision_scores))
    except Exception:
        y_proba = calibrated_model.predict(X_test).astype(float)


In [93]:
# ===== 3) THRESHOLDING (choose operating point) & EVALUATION =====
# NOTE: this cell picks the threshold on TEST (diagnostic / leaky). Use for EDA only.

# Sweep thresholds on TEST (diagnostic) → pick best by macro-F1
threshold_results = evaluate_threshold_sweep(y_test, y_proba, objective="macro_f1")
best_row = threshold_results.iloc[0].to_dict()
best_threshold = float(best_row["threshold"])

print(f"Best threshold (by macro-F1): {best_threshold:.2f}")
print({k: round(float(best_row[k]), 4) for k in ["precision","recall","f1","macro_f1","ap","roc_auc","brier"]})

# Predictions at chosen threshold
y_pred = (y_proba >= best_threshold).astype(int)

# ---------- SAVE DIAGNOSTIC ARTIFACTS ----------
diag_dir = (run_dir / "figures" / f"diag_{train_session_name}_to_{test_session_name}__{selected_model_name}")
diag_dir.mkdir(parents=True, exist_ok=True)

# Full sweep table (so future-you can audit the leakage)
threshold_results.to_csv(diag_dir / "threshold_sweep_on_TEST.csv", index=False)

# Save arrays to allow re-plotting without retraining
np.save(diag_dir / "y_true.npy", y_test)
np.save(diag_dir / "y_proba.npy", y_proba)
np.save(diag_dir / "y_pred_at_best_thr.npy", y_pred)

# Tiny metrics summary at chosen threshold
with open(diag_dir / "metrics_at_best_thr.json", "w") as f:
    json.dump({
        "threshold": best_threshold,
        "precision": float(best_row["precision"]),
        "recall":    float(best_row["recall"]),
        "f1":        float(best_row["f1"]),
        "macro_f1":  float(best_row["macro_f1"]),
        "ap":        float(best_row["ap"]),
        "roc_auc":   float(best_row["roc_auc"]),
        "brier":     float(best_row["brier"]),
        "note": "diagnostic: threshold optimized on TEST (leaky; not for final reporting)"
    }, f, indent=2)

# Plots (confusion matrices, PR curves, calibration) → saved to disk
# save_all_eval_figures expects (N, C) probs; convert binary p → (1-p, p)
y_proba_2d = np.vstack([1 - y_proba, y_proba]).T
save_all_eval_figures(diag_dir, y_true=y_test, y_pred=y_pred, y_proba=y_proba_2d, class_names=CLASS_NAMES)

print(f"[INFO] Diagnostic figs & tables saved → {diag_dir}")


Best threshold (by macro-F1): 0.45
{'precision': 0.6429, 'recall': 0.4696, 'f1': 0.5427, 'macro_f1': 0.4956, 'ap': 0.6073, 'roc_auc': 0.4589, 'brier': 0.2839}
[INFO] Diagnostic figs & tables saved → /Users/kwill55/RVL/personalized-sports-optimization/experiments/2025-08-19_20-48-21_run-001_baseline/figures/diag_session_02_to_session_03__LogisticRegression


In [94]:
# ===== 4) DIAGNOSTIC RUNNER: loop models & directions (threshold picked on test) =====
results_rows = []

for train_session_name, test_session_name in [("session_02","session_03"), ("session_03","session_02")]:
    # Load & align
    X_train_df, y_train, out_dir_train = None, None, None
    X_test_df,  y_test,  out_dir_test  = None, None, None
    for session_name, (X_path, y_path, out_root) in SESSIONS.items():
        X_df, y = load_features_and_labels(X_path, y_path)
        if session_name == train_session_name: X_train_df, y_train, out_dir_train = X_df, y, out_root
        if session_name == test_session_name:  X_test_df,  y_test,  out_dir_test  = X_df, y, out_root

    X_train_df, X_test_df, shared_feature_cols, meta_cols = align_feature_columns(X_train_df, X_test_df)
    X_train, X_train_meta = split_features_and_metadata(X_train_df, shared_feature_cols, meta_cols)
    X_test,  X_test_meta  = split_features_and_metadata(X_test_df,  shared_feature_cols, meta_cols)

    for model_name, base_estimator in MODELS.items():
        # class_weight path
        use_class_weight = (REBALANCE == "class_weight") and hasattr(base_estimator, "set_params")
        if use_class_weight:
            try: base_estimator = base_estimator.set_params(class_weight="balanced")
            except Exception: pass

        # train
        training_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", base_estimator)])
        trained_pipeline = training_pipeline.fit(X_train, y_train)

        # calibrate
        if CALIBRATE:
            minority_count = min(np.bincount(y_train)) if len(np.unique(y_train)) == 2 else 0
            n_splits = 3 if minority_count >= 3 else 2
            calibration_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            calibrated_model = CalibratedClassifierCV(estimator=trained_pipeline, method=CALIB_METHOD, cv=calibration_cv).fit(X_train, y_train)
        else:
            calibrated_model = trained_pipeline

        # proba on test
        if hasattr(calibrated_model, "predict_proba"):
            y_proba = calibrated_model.predict_proba(X_test)[:, 1]
        else:
            try:
                y_proba = 1/(1+np.exp(-calibrated_model.decision_function(X_test)))
            except Exception:
                y_proba = calibrated_model.predict(X_test).astype(float)

        # threshold sweep (diagnostic: on test)
        sweep = evaluate_threshold_sweep(y_test, y_proba, objective="macro_f1")
        best = sweep.iloc[0].to_dict()
        thr = float(best["threshold"])
        y_pred = (y_proba >= thr).astype(int)

        # --- SAVE: diagnostic artifacts ---
        diag_dir = (run_dir / "figures" / f"diag_{train_session_name}_to_{test_session_name}__{model_name}")
        diag_dir.mkdir(parents=True, exist_ok=True)

        # Save the entire threshold sweep done on TEST (diagnostic/leaky by design)
        sweep.to_csv(diag_dir / "threshold_sweep_on_TEST.csv", index=False)

        # Save figures (convert binary proba to (N,2) matrix)
        y_proba_2d = np.vstack([1 - y_proba, y_proba]).T
        save_all_eval_figures(diag_dir, y_true=y_test, y_pred=y_pred, y_proba=y_proba_2d, class_names=CLASS_NAMES)

        # record
        results_rows.append({
            "train_session": train_session_name,
            "test_session":  test_session_name,
            "model":          model_name,
            "rebalance":      REBALANCE,
            "calibrated":     CALIBRATE,
            "thr":            thr,
            **{k: float(best[k]) for k in ["precision","recall","f1","macro_f1","ap","roc_auc","brier"]}
        })

# leaderboard (SAVE)
diagnostic_results = pd.DataFrame(results_rows).sort_values(["macro_f1","f1","ap"], ascending=False)
diagnostic_results.to_csv(run_dir / "diagnostic_results.csv", index=False)
diagnostic_results


,train_session,test_session,model,rebalance,calibrated,thr,precision,recall,f1,macro_f1,ap,roc_auc,brier
3,session_02,session_03,GradientBoosting,class_weight,True,0.60,0.672269,0.695652,0.683761,0.557265,0.648803,0.532771,0.232223
7,session_03,session_02,RandomForest,class_weight,True,0.60,0.674699,0.861538,0.756757,0.518378,0.667865,0.519457,0.225644
2,session_02,session_03,RandomForest,class_weight,True,0.55,0.645833,0.539130,0.587678,0.509525,0.631926,0.504867,0.241236
0,session_02,session_03,LogisticRegression,class_weight,True,0.45,0.642857,0.469565,0.542714,0.495599,0.607350,0.458923,0.283916
8,session_03,session_02,GradientBoosting,class_weight,True,0.65,0.777778,0.323077,0.456522,0.492412,0.797152,0.652941,0.220829
6,session_03,session_02,LinearSVC,class_weight,True,0.65,0.674419,0.446154,0.537037,0.490741,0.674330,0.508145,0.228611
5,session_03,session_02,LogisticRegression,class_weight,True,0.60,0.645161,0.615385,0.629921,0.483975,0.663716,0.476471,0.232248
1,session_02,session_03,LinearSVC,class_weight,True,0.50,0.626506,0.452174,0.525253,0.479494,0.611235,0.460221,0.274751
9,session_03,session_02,GaussianNB,class_weight,True,0.45,0.635135,0.723077,0.676259,0.456774,0.602489,0.401357,0.248678
4,session_02,session_03,GaussianNB,class_weight,True,0.05,0.631868,1.000000,0.774411,0.387205,0.629272,0.489422,0.234861


In [95]:
# ===== 5) RIGOROUS RUNNER: choose threshold via inner CV on TRAIN, then test once =====
def pick_threshold_via_cv(model_pipeline, X_train, y_train, objective="macro_f1", method="sigmoid"):
    """Return (calibrated_model, chosen_threshold) using inner CV on training data only."""
    minority_count = min(np.bincount(y_train)) if len(np.unique(y_train)) == 2 else 0
    n_splits = 3 if minority_count >= 3 else 2
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Get out-of-fold probabilities for threshold selection
    oof_proba = np.zeros_like(y_train, dtype=float)
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        # train a fresh copy
        pipe_copy = Pipeline(model_pipeline.steps)
        # calibrate within the fold
        calibrated = CalibratedClassifierCV(estimator=pipe_copy, method=method, cv=2).fit(X_tr, y_tr)
        if hasattr(calibrated, "predict_proba"):
            oof_proba[val_idx] = calibrated.predict_proba(X_val)[:, 1]
        else:
            try:
                oof_proba[val_idx] = 1/(1+np.exp(-calibrated.decision_function(X_val)))
            except Exception:
                oof_proba[val_idx] = calibrated.predict(X_val).astype(float)

    # choose threshold on OOF
    sweep = evaluate_threshold_sweep(y_train, oof_proba, objective=objective)
    thr = float(sweep.iloc[0]["threshold"])

    # now refit on full training data (with calibration) to deploy
    final_calibrated = CalibratedClassifierCV(estimator=Pipeline(model_pipeline.steps), method=method, cv=cv).fit(X_train, y_train)
    return final_calibrated, thr

rigorous_rows = []

for train_session_name, test_session_name in [("session_02","session_03"), ("session_03","session_02")]:
    # load & align
    X_train_df, y_train, out_dir_train = None, None, None
    X_test_df,  y_test,  out_dir_test  = None, None, None
    for session_name, (X_path, y_path, out_root) in SESSIONS.items():
        X_df, y = load_features_and_labels(X_path, y_path)
        if session_name == train_session_name: X_train_df, y_train, out_dir_train = X_df, y, out_root
        if session_name == test_session_name:  X_test_df,  y_test,  out_dir_test  = X_df, y, out_root

    X_train_df, X_test_df, shared_feature_cols, meta_cols = align_feature_columns(X_train_df, X_test_df)
    X_train, X_train_meta = split_features_and_metadata(X_train_df, shared_feature_cols, meta_cols)
    X_test,  X_test_meta  = split_features_and_metadata(X_test_df,  shared_feature_cols, meta_cols)

    for model_name, base_estimator in MODELS.items():
        # class weights if requested
        use_class_weight = (REBALANCE == "class_weight") and hasattr(base_estimator, "set_params")
        if use_class_weight:
            try: base_estimator = base_estimator.set_params(class_weight="balanced")
            except Exception: pass

        model_pipeline = Pipeline([("scaler", StandardScaler()), ("clf", base_estimator)])

        # pick threshold via train-CV (and get calibrated model)
        calibrated_model, chosen_thr = pick_threshold_via_cv(
            model_pipeline, X_train, y_train, objective="macro_f1", method=CALIB_METHOD
        )

        # evaluate once on test at fixed threshold
        if hasattr(calibrated_model, "predict_proba"):
            y_proba = calibrated_model.predict_proba(X_test)[:, 1]
        else:
            try:
                y_proba = 1/(1+np.exp(-calibrated_model.decision_function(X_test)))
            except Exception:
                y_proba = calibrated_model.predict(X_test).astype(float)
        y_pred = (y_proba >= chosen_thr).astype(int)

        # metrics
        p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
        per_class = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)
        macro_f1 = float(np.mean(per_class[2]))
        ap = average_precision_score(y_test, y_proba)
        roc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test))==2 else np.nan
        brier = brier_score_loss(y_test, y_proba)

        # --- SAVE: rigorous artifacts ---
        rig_dir = (run_dir / "figures" / f"rig_{train_session_name}_to_{test_session_name}__{model_name}")
        rig_dir.mkdir(parents=True, exist_ok=True)

        # Record the locked threshold + method for auditability
        with open(rig_dir / "chosen_threshold.txt", "w") as f:
            f.write(f"threshold={chosen_thr:.6f}\nmethod={CALIB_METHOD}\nobjective=macro_f1\n")

        # Plots at locked threshold
        y_proba_2d = np.vstack([1 - y_proba, y_proba]).T
        save_all_eval_figures(rig_dir, y_true=y_test, y_pred=y_pred, y_proba=y_proba_2d, class_names=CLASS_NAMES)

        rigorous_rows.append({
            "train_session": train_session_name,
            "test_session":  test_session_name,
            "model":          model_name,
            "rebalance":      REBALANCE,
            "calibrated":     True,
            "thr":            chosen_thr,
            "precision":      float(p),
            "recall":         float(r),
            "f1":             float(f1),
            "macro_f1":       float(macro_f1),
            "ap":             float(ap),
            "roc_auc":        float(roc),
            "brier":          float(brier),
        })

rigorous_results = pd.DataFrame(rigorous_rows).sort_values(["macro_f1","f1","ap"], ascending=False)
rigorous_results.to_csv(run_dir / "rigorous_results.csv", index=False)
rigorous_results


,train_session,test_session,model,rebalance,calibrated,thr,precision,recall,f1,macro_f1,ap,roc_auc,brier
3,session_02,session_03,GradientBoosting,class_weight,True,0.60,0.672269,0.695652,0.683761,0.557265,0.648803,0.532771,0.232223
7,session_03,session_02,RandomForest,class_weight,True,0.60,0.674699,0.861538,0.756757,0.518378,0.667865,0.519457,0.225644
5,session_03,session_02,LogisticRegression,class_weight,True,0.60,0.645161,0.615385,0.629921,0.483975,0.663716,0.476471,0.232248
6,session_03,session_02,LinearSVC,class_weight,True,0.60,0.643836,0.723077,0.681159,0.473913,0.674330,0.508145,0.228611
8,session_03,session_02,GradientBoosting,class_weight,True,0.60,0.641304,0.907692,0.751592,0.400186,0.797152,0.652941,0.220829
2,session_02,session_03,RandomForest,class_weight,True,0.65,0.562500,0.078261,0.137405,0.326213,0.631926,0.504867,0.241236
0,session_02,session_03,LogisticRegression,class_weight,True,0.65,0.583333,0.060870,0.110236,0.316721,0.607350,0.458923,0.283916
1,session_02,session_03,LinearSVC,class_weight,True,0.65,0.444444,0.069565,0.120301,0.306904,0.611235,0.460221,0.274753
4,session_02,session_03,GaussianNB,class_weight,True,0.65,0.000000,0.000000,0.000000,0.269076,0.629272,0.489422,0.234861
9,session_03,session_02,GaussianNB,class_weight,True,0.60,0.000000,0.000000,0.000000,0.255639,0.602489,0.401357,0.248678
